# M3L4 E01 — MiniTracer en Python puro [OK] Resolution
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

## ¿Por qué importa este ejercicio?

En el ejercicio anterior vimos cómo un **trace estructurado** permite responder preguntas que los logs no pueden. Pero escribir traces a mano como diccionarios es tedioso y propenso a errores.

Acá vamos a construir un **MiniTracer**: una clase Python que automatiza:
- La creación de traces con IDs únicos
- La agregación de spans con input, output y duración
- El cálculo automático de tiempo total

Este MiniTracer es una versión simplificada de cómo funcionan herramientas de tracing reales como **Langfuse**, **LangSmith** o **Weights & Biases** por debajo.

| Concepto | Definición simple | Cómo aparece en MiniTracer |
|---|---|---|
| **Trace** | Ciclo de vida completo de una request | `start_trace()` crea el contenedor con `trace_id` |
| **Span** | Paso individual dentro del trace | `add_span()` agrega un paso con nombre, input, output y duración |
| **Generation** | Llamada a un LLM dentro de un span | No implementado en MiniTracer (se agrega como span con metadata) |
| **Trace ID** | Identificador único para correlacionar eventos | `uuid.uuid4()` generado automáticamente |
| **Duración** | Tiempo total del trace | Calculado en `update_trace_output()` |

In [ ]:
import time
import uuid
from datetime import datetime
from typing import Optional, Any

## Solución — Clase MiniTracer

La clase tiene 5 métodos que cubren el ciclo completo de tracing:

1. **`start_trace()`** — Inicia un trace con nombre, input, metadata, tags y timestamp
2. **`add_span()`** — Agrega un paso con su input, output y duración
3. **`update_trace_output()`** — Fija el output final y calcula la duración total
4. **`show_trace()`** — Devuelve el trace completo para inspección
5. **`get_all_traces()`** — Lista todos los traces almacenados

> **¿Por qué `_started_at_ts` tiene guión bajo?** Es un campo interno que el usuario no debería tocar. En Python, el guión bajo indica "privado" por convención.

In [ ]:
class MiniTracer:
    def __init__(self):
        self.traces = []

    def start_trace(self, name: str, input_data=None, metadata=None, tags=None) -> dict:
        trace = {
            'trace_id': str(uuid.uuid4()),
            'name': name,
            'input': input_data,
            'output': None,
            'metadata': metadata or {},
            'tags': tags or [],
            'spans': [],
            'created_at': datetime.utcnow().isoformat(),
            '_started_at_ts': time.time()   # para calcular duración total
        }
        self.traces.append(trace)
        return trace

    def add_span(self, trace: dict, name: str, input_data=None,
                 output_data=None, metadata=None, duration_ms=None) -> dict:
        span = {
            'span_id': str(uuid.uuid4()),
            'name': name,
            'input': input_data,
            'output': output_data,
            'metadata': metadata or {},
            'duration_ms': duration_ms
        }
        trace['spans'].append(span)
        return span

    def update_trace_output(self, trace: dict, output_data: Any) -> None:
        trace['output'] = output_data
        elapsed = round((time.time() - trace.get('_started_at_ts', time.time())) * 1000, 2)
        trace['total_duration_ms'] = elapsed

    def show_trace(self, trace: dict) -> dict:
        return trace

    def get_all_traces(self) -> list:
        return self.traces


print('MiniTracer definido.')

## Uso del MiniTracer

Simulemos el mismo caso del ejercicio anterior: una consulta de factura que pasa por routing y luego por el agente de finanzas.

Cada llamada a `add_span()` registra un paso. Al final, `update_trace_output()` cierra el trace y calcula la duración total automáticamente.

In [ ]:
tracer = MiniTracer()

trace = tracer.start_trace(
    name='support-request',
    input_data={'query': 'No puedo ver mi factura'},
    metadata={'environment': 'notebook', 'user_id': 'student-01'},
    tags=['demo', 'm3l4']
)

tracer.add_span(
    trace,
    name='orchestrator-routing',
    input_data={'query': 'No puedo ver mi factura'},
    output_data={'intent': 'finance'},
    duration_ms=120
)

tracer.add_span(
    trace,
    name='finance-agent',
    input_data={'query': 'No puedo ver mi factura'},
    output_data={'answer': 'Podés ver tu factura desde el portal de pagos.'},
    duration_ms=840
)

tracer.update_trace_output(
    trace,
    {'final_answer': 'Podés ver tu factura desde el portal de pagos.'}
)

tracer.show_trace(trace)

## Verificación

Corremos asserts para validar que el MiniTracer funciona correctamente:
- El trace tiene `trace_id`, `name`, `spans`, `input`, `output`, `metadata`, `tags`
- Hay exactamente 2 spans (routing + agente)
- El output no es `None`
- El tag 'm3l4' está presente
- La duración total se calculó automáticamente

In [ ]:
assert 'trace_id' in trace
assert 'name' in trace
assert 'spans' in trace
assert 'input' in trace
assert 'output' in trace
assert 'metadata' in trace
assert 'tags' in trace
assert len(trace['spans']) == 2, f'Se esperaban 2 spans, hay {len(trace["spans"])}'
assert trace['output'] is not None
assert 'm3l4' in trace['tags']
assert 'total_duration_ms' in trace

print('Checks E01 OK')
print(f"Trace ID: {trace['trace_id']}")
print(f"Spans: {[s['name'] for s in trace['spans']]}")
print(f"Total duration: {trace['total_duration_ms']} ms")

## [OK] Cierre — ¿Qué logramos?

| Antes (E00) | Ahora (E01) |
|---|---|
| Traces escritos a mano como diccionarios | Traces creados con `MiniTracer.start_trace()` |
| Sin IDs únicos | `trace_id` y `span_id` automáticos con UUID |
| Duración calculada manualmente | `total_duration_ms` calculado automáticamente |
| Sin estructura reusable | Clase `MiniTracer` reusable en cualquier sistema |

**¿Qué sigue?** En E02 vamos a integrar este MiniTracer dentro de un sistema multiagente real para tracear cada request automáticamente.